# MWE Filter Evaluation Analysis (HIMYM s04e12 l1055-1259)

## Introduction
This notebook compares phrasal-verb filter strategies on the same gold slice and shows overall metrics plus exact TP/FP/FN examples per filter.

## What Each Filter Means
Filter behavior is implemented in `app/search/mwe/phrasal_verbs.py` (`PV_FILTERS`).

- **none**: no filtering; accepts every lemma match (baseline).
- **dep_only**: strict dependency filter; keeps only matches where spaCy finds a `prt` particle attached to the verb.
- **blocklist**: rule-based blocklist; rejects known high-FP literal combinations (e.g., `go to`, `be in`) and keeps everything else.
- **dep_extended**: accepts `prt` and `advmod`; for `prep`, rejects cases with `pobj` when the verb is high-frequency, otherwise accepts.
- **dep_highfreq**: applies strict `prt` filtering only for high-frequency verbs; non-high-frequency verbs are accepted without dep filtering.
- **hybrid**: blocklist-first plus dependency-aware fallback; blocklisted pairs pass only if true `prt` is found, non-blocklisted pairs are mostly permissive (accepts `prt`/`advmod`/`prep`, with permissive fallback).

## Output Sections
1. Metrics comparison table across filters.
2. Per-filter breakdown: False Positives, False Negatives, and True Positives.

In [4]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 220)

In [5]:
DATASET_DIR = Path('../manual_tests/datasets/himym/s04e12_benefits_l1055_1259')
RUNS_DIR = DATASET_DIR / 'runs'
GOLD_PATH = DATASET_DIR / 'gold.csv'

PREFERRED_ORDER = ['none', 'dep_only', 'blocklist', 'dep_extended', 'dep_highfreq', 'hybrid']

def extract_filter_name(path: Path) -> str | None:
    m = re.search(r'_filter_([^_]+(?:_[^_]+)*)_predictions\.csv$', path.name)
    return m.group(1) if m else None

def to_lookup(df: pd.DataFrame) -> dict:
    lookup = {}
    for _, row in df.iterrows():
        lookup[row['key']] = row.to_dict()
    return lookup

gold_df = pd.read_csv(GOLD_PATH)
gold_df['line_index'] = gold_df['line_index'].astype(int)
gold_df['key'] = list(zip(gold_df['line_index'], gold_df['expression_type'], gold_df['expression']))
gold_df = gold_df.drop_duplicates(subset=['key']).reset_index(drop=True)
gold_lookup = to_lookup(gold_df)
gold_keys = set(gold_lookup.keys())

prediction_files = sorted(RUNS_DIR.glob('*_filter_*_predictions.csv'))
filters = {}
for pred_path in prediction_files:
    name = extract_filter_name(pred_path)
    if not name:
        continue
    pred_df = pd.read_csv(pred_path)
    pred_df['line_index'] = pred_df['line_index'].astype(int)
    pred_df['key'] = list(zip(pred_df['line_index'], pred_df['expression_type'], pred_df['expression']))
    pred_df = pred_df.drop_duplicates(subset=['key']).reset_index(drop=True)
    filters[name] = pred_df

ordered_filters = [f for f in PREFERRED_ORDER if f in filters] + [f for f in filters if f not in PREFERRED_ORDER]
print('Loaded filters:', ordered_filters)
print('Gold instances:', len(gold_keys))

Loaded filters: ['none', 'dep_only', 'blocklist', 'dep_extended', 'dep_highfreq', 'hybrid']
Gold instances: 18


In [6]:
def details_df(keys, source_lookup):
    if not keys:
        return pd.DataFrame(columns=['line_index', 'expression_type', 'expression', 'speaker', 'line'])

    rows = []
    for k in sorted(keys):
        row = source_lookup[k]
        rows.append({
            'line_index': int(row['line_index']),
            'expression_type': row['expression_type'],
            'expression': row['expression'],
            'speaker': row.get('speaker', ''),
            'line': row.get('line', ''),
        })

    return pd.DataFrame(rows).sort_values(['line_index', 'expression_type', 'expression']).reset_index(drop=True)

results = {}
metric_rows = []

for filter_name in ordered_filters:
    pred_df = filters[filter_name]
    pred_lookup = to_lookup(pred_df)
    pred_keys = set(pred_lookup.keys())

    tp = gold_keys & pred_keys
    fp = pred_keys - gold_keys
    fn = gold_keys - pred_keys

    precision = len(tp) / len(pred_keys) if pred_keys else 0.0
    recall = len(tp) / len(gold_keys) if gold_keys else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    metric_rows.append({
        'Filter': filter_name,
        'Pred': len(pred_keys),
        'TP': len(tp),
        'FP': len(fp),
        'FN': len(fn),
        'Precision': round(precision, 3),
        'Recall': round(recall, 3),
        'F1': round(f1, 3),
    })

    results[filter_name] = {
        'tp': details_df(tp, gold_lookup),
        'fp': details_df(fp, pred_lookup),
        'fn': details_df(fn, gold_lookup),
    }

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

,Filter,Pred,TP,FP,FN,Precision,Recall,F1
0,none,31,14,17,4,0.452,0.778,0.571
1,dep_only,10,9,1,9,0.900,0.500,0.643
2,blocklist,22,14,8,4,0.636,0.778,0.700
3,dep_extended,16,12,4,6,0.750,0.667,0.706
4,dep_highfreq,17,12,5,6,0.706,0.667,0.686
5,hybrid,23,14,9,4,0.609,0.778,0.683


In [7]:
for filter_name in ordered_filters:
    print(f'\n=== FILTER: {filter_name} ===')

    filter_metrics = metrics_df[metrics_df['Filter'] == filter_name].reset_index(drop=True)
    print('Metrics')
    display(filter_metrics)

    print('False Positives')
    display(results[filter_name]['fp'])

    print('False Negatives')
    display(results[filter_name]['fn'])

    print('True Positives')
    display(results[filter_name]['tp'])


=== FILTER: none ===
Metrics


,Filter,Pred,TP,FP,FN,Precision,Recall,F1
0,none,31,14,17,4,0.452,0.778,0.571


False Positives


,line_index,expression_type,expression,speaker,line
0,14,phrasal_verb,pass on,Joey,"Yeah, I've never been here before. Just, uh, passing through on my way to Hartford."
1,14,phrasal_verb,pass on to,Joey,"Yeah, I've never been here before. Just, uh, passing through on my way to Hartford."
2,53,phrasal_verb,get to,Lorelai,Getting to know my daughter.
3,72,phrasal_verb,look to,Drella,"Okay. I am a great harp player, and this is my great harp, okay. So if you're looking for someone to just be nice to the guests, get a harmonica player, or maybe some guy who whistles through his ..."
4,96,phrasal_verb,go to,Rory,"When are you going to let your parents know that you listen to the evil rock music? You're an American teenager, for God's sake."
5,100,phrasal_verb,go to,Lane,My parents set me up with the son of a business associate. He's gonna be a doctor.
6,138,phrasal_verb,get on,Lorelai,"All right. So, now, let's get you up and to the doctor, on three. One-two-three."
7,138,phrasal_verb,get to,Lorelai,"All right. So, now, let's get you up and to the doctor, on three. One-two-three."
8,138,phrasal_verb,get up to,Lorelai,"All right. So, now, let's get you up and to the doctor, on three. One-two-three."
9,138,phrasal_verb,let up,Lorelai,"All right. So, now, let's get you up and to the doctor, on three. One-two-three."


False Negatives


...

,line_index,expression_type,expression,speaker,line
0,14,phrasal_verb,pass on,Joey,"Yeah, I've never been here before. Just, uh, passing through on my way to Hartford."
1,14,phrasal_verb,pass on to,Joey,"Yeah, I've never been here before. Just, uh, passing through on my way to Hartford."
2,138,phrasal_verb,get on,Lorelai,"All right. So, now, let's get you up and to the doctor, on three. One-two-three."
3,138,phrasal_verb,get up to,Lorelai,"All right. So, now, let's get you up and to the doctor, on three. One-two-three."
4,138,phrasal_verb,let up,Lorelai,"All right. So, now, let's get you up and to the doctor, on three. One-two-three."
5,178,phrasal_verb,get out,Mrs. Kim,"KIM: Go upstairs. Tea is ready. I have muffins - no dairy, no sugar, no wheat. You have to soak them in tea to make them soft enough to bite but they're very healthy. So, how was school? None of t..."
6,178,phrasal_verb,get out!,Mrs. Kim,"KIM: Go upstairs. Tea is ready. I have muffins - no dairy, no sugar, no wheat. You have to soak them in tea to make them soft enough to bite but they're very healthy. So, how was school? None of t..."
7,178,phrasal_verb,soak in,Mrs. Kim,"KIM: Go upstairs. Tea is ready. I have muffins - no dairy, no sugar, no wheat. You have to soak them in tea to make them soft enough to bite but they're very healthy. So, how was school? None of t..."
8,180,phrasal_verb,come of,Rory,"Though come to think of it, Joanna Posner was glowing a little."


False Negatives


,line_index,expression_type,expression,speaker,line
0,24,phrasal_verb,get going,Joey,So I guess I should get going.
1,27,phrasal_verb,screw with,Lorelai,"I'm just screwing with your mind, Joey. It's nice to meet you. Enjoy Hartford."
2,53,phrasal_verb,get to know,Lorelai,Getting to know my daughter.
3,104,phrasal_verb,plan ahead,Lane,"Well, my parents like to plan ahead."


True Positives


,line_index,expression_type,expression,speaker,line
0,14,phrasal_verb,pass through,Joey,"Yeah, I've never been here before. Just, uh, passing through on my way to Hartford."
1,18,phrasal_verb,sit down,Joey,"Hey, you mind if I sit down?"
2,72,phrasal_verb,look for,Drella,"Okay. I am a great harp player, and this is my great harp, okay. So if you're looking for someone to just be nice to the guests, get a harmonica player, or maybe some guy who whistles through his ..."
3,75,phrasal_verb,attend to,Lorelai,Has the plumber attended to room four yet?
4,77,phrasal_verb,talk to,Lorelai,"Hi Marco, Lorelai. Talk to me about room four. What was wrong with it?"
5,86,phrasal_verb,look at,Lorelai,"Ooh, hey, have Michel look at your French paper before you go."
6,96,idiom,for god sake,Rory,"When are you going to let your parents know that you listen to the evil rock music? You're an American teenager, for God's sake."
7,100,phrasal_verb,set up,Lane,My parents set me up with the son of a business associate. He's gonna be a doctor.
8,110,phrasal_verb,clear up,Lane,Don't expect me to clear it up for you.
9,134,phrasal_verb,line up,Lorelai,"Someday when we open our own inn, diabetics will line up to eat this sauce."
